In [1]:
import pandas as pd
import numpy as np
import gseapy as gp
import pickle
import mygene


In [5]:
for i in gp.get_library_name(organism='Human'):
    if '2019' in i.lower():
        print(i)

BioPlanet_2019
ClinVar_2019
DepMap_WG_CRISPR_Screens_Broad_CellLines_2019
DepMap_WG_CRISPR_Screens_Sanger_CellLines_2019
GWAS_Catalog_2019
InterPro_Domains_2019
KEGG_2019_Human
KEGG_2019_Mouse
Pfam_Domains_2019
PheWeb_2019
TRRUST_Transcription_Factors_2019
WikiPathways_2019_Human
WikiPathways_2019_Mouse


In [11]:
def to_pathway_df(db):
    keggs_2019 =gp.get_library(name=db,organism='human')
    all_genes = sorted(set(g for genes in keggs_2019.values() for g in genes))

    # create matrix
    df = pd.DataFrame(0, index=all_genes, columns=keggs_2019.keys())
    for col, genes in keggs_2019.items():
        df.loc[genes, col] = 1
    return df

In [14]:
df = to_pathway_df('KEGG_2019_Human')
df.shape

(7802, 308)

In [12]:
wiki_df = to_pathway_df('WikiPathways_2019_Human')

In [13]:
wiki_df.shape

(6201, 472)

In [15]:
ppi = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/stringdb/uniport_ppi_2019.csv')

In [16]:
ppi.head(3)

,string_id,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_119,feature_120,feature_121,feature_122,feature_123,feature_124,feature_125,feature_126,feature_127,feature_128
0,P84085,-0.188154,0.192034,0.021007,-0.115466,-0.106862,0.001464,0.079977,0.308825,-0.013989,...,0.163154,0.025956,-0.154160,0.047249,0.013052,-0.121823,0.261028,0.076985,0.027698,0.038653
1,P05129,-0.056868,0.169329,-0.159240,0.109760,-0.025587,0.082301,0.017834,0.020714,-0.020855,...,0.243663,-0.131249,-0.205958,-0.202946,-0.267179,-0.182676,-0.227924,0.124403,-0.201234,-0.194766
2,Q9UQR1,-0.055811,0.039503,0.155353,0.067647,0.073602,0.088512,0.010504,0.194107,0.187011,...,-0.076926,-0.093258,-0.206983,-0.309286,0.032149,0.053479,0.090693,0.144785,0.106935,0.007952


In [17]:

mg = mygene.MyGeneInfo()
# Query mygene for UniProt and Entrez gene ID mappings
results = mg.querymany(
    df.index.tolist(),
    scopes='symbol',
    fields='uniprot,entrezgene',
    species='human'
)

INFO:biothings.client:querying 1-1000 ...
INFO:biothings.client:querying 1001-2000 ...
INFO:biothings.client:querying 2001-3000 ...
INFO:biothings.client:querying 3001-4000 ...
INFO:biothings.client:querying 4001-5000 ...
INFO:biothings.client:querying 5001-6000 ...
INFO:biothings.client:querying 6001-7000 ...
INFO:biothings.client:querying 7001-7802 ...
INFO:biothings.client:Finished.
INFO:biothings.client:Pass "returnall=True" to return complete lists of duplicate or missing query terms.


In [18]:
results_df = pd.DataFrame(results)
results_df = results_df[~results_df['entrezgene'].isna()]
results_df['uniprot_ids'] = results_df['uniprot'].apply(
    lambda x: list(x.values())[0] if isinstance(x, dict) and 'Swiss-Prot' in x else None)
results_df = results_df[~results_df['uniprot_ids'].isna()]
results_df[results_df['uniprot_ids'].apply(lambda x: isinstance(x, list) and len(x) > 1)]

,query,_id,_score,entrezgene,uniprot,notfound,uniprot_ids
329,AMY1A,276,25.374277,276,"{'Swiss-Prot': ['P0DTE7', 'P0DUB6', 'P0DTE8'],...",NaN,"[P0DTE7, P0DUB6, P0DTE8]"
330,AMY1B,277,27.387224,277,"{'Swiss-Prot': ['P0DTE7', 'P0DUB6', 'P0DTE8'],...",NaN,"[P0DTE7, P0DUB6, P0DTE8]"
331,AMY1C,278,22.482887,278,"{'Swiss-Prot': ['P0DTE7', 'P0DUB6', 'P0DTE8'],...",NaN,"[P0DTE7, P0DUB6, P0DTE8]"
644,BBC3,27113,18.347180,27113,"{'Swiss-Prot': ['Q9BXH1', 'Q96PG8']}",NaN,"[Q9BXH1, Q96PG8]"
761,C4B,721,19.491293,721,"{'Swiss-Prot': ['P0C0L5', 'P0C0L4'], 'TrEMBL':...",NaN,"[P0C0L5, P0C0L4]"
819,CALCA,796,18.890875,796,"{'Swiss-Prot': ['P01258', 'P06881']}",NaN,"[P01258, P06881]"
826,CALM1,801,18.288290,801,"{'Swiss-Prot': ['P0DP24', 'P0DP23', 'P0DP25'],...",NaN,"[P0DP24, P0DP23, P0DP25]"
827,CALM2,805,18.274733,805,"{'Swiss-Prot': ['P0DP24', 'P0DP23', 'P0DP25'],...",NaN,"[P0DP24, P0DP23, P0DP25]"
828,CALM3,808,18.869629,808,"{'Swiss-Prot': ['P0DP24', 'P0DP23', 'P0DP25'],...",NaN,"[P0DP24, P0DP23, P0DP25]"
923,CCL4L1,388372,26.595705,388372,"{'Swiss-Prot': ['P13236', 'Q8NHW4'], 'TrEMBL':...",NaN,"[P13236, Q8NHW4]"


In [19]:
len(results_df['query'].unique())

7400

In [20]:
multi_map_ids = results_df[results_df['uniprot_ids'].apply(lambda x: isinstance(x, list) and len(x) > 1)]['query']

In [21]:
multi_map_uniport_ids = results_df[results_df['uniprot_ids'].apply(lambda x: isinstance(x, list) and len(x) > 1)]['uniprot_ids']

In [22]:
unique_items = set(item for sublist in multi_map_uniport_ids for item in sublist)

In [23]:
len(unique_items)

46

In [24]:
ppi[ppi['string_id'].isin(unique_items)]['string_id'].tolist()

['P0DMM9', 'P0C0L4', 'P50391']

In [25]:
targets = {'P0DMM9', 'P0C0L4', 'P50391'}

results_df[
    results_df['uniprot_ids'].apply(
        lambda x: isinstance(x, list) and any(i in targets for i in x)
    )
]

,query,_id,_score,entrezgene,uniprot,notfound,uniprot_ids
761,C4B,721,19.491293,721,"{'Swiss-Prot': ['P0C0L5', 'P0C0L4'], 'TrEMBL':...",NaN,"[P0C0L5, P0C0L4]"
4365,NPY4R2,100996758,19.823011,100996758,"{'Swiss-Prot': ['P50391', 'P0DQD5']}",NaN,"[P50391, P0DQD5]"
6742,SULT1A4,445329,27.388079,445329,"{'Swiss-Prot': ['P0DMN0', 'P0DMM9'], 'TrEMBL':...",NaN,"[P0DMN0, P0DMM9]"


In [26]:
multi_map_filter_dict = {
    'SULT1A4':'P0DMM9', 
    'C4B':'P0C0L4', 
    'NPY4R2':'P50391'}

In [27]:
filter_map_df = results_df[~results_df['uniprot_ids'].apply(lambda x: isinstance(x, list) and len(x) > 1)]

In [28]:
multi_map_filter_dict.update(
    dict(zip(filter_map_df['query'], filter_map_df['uniprot_ids']))
)

In [29]:
uniport_kegg_df = df.copy()

# map gene → uniprot_ids
uniport_kegg_df['uniprot_ids'] = uniport_kegg_df.index.map(multi_map_filter_dict)

# drop rows with no mapping
uniport_kegg_df = uniport_kegg_df.dropna(subset=['uniprot_ids'])

In [30]:
uniport_kegg_df

,ABC transporters,AGE-RAGE signaling pathway in diabetic complications,AMPK signaling pathway,Acute myeloid leukemia,Adherens junction,Adipocytokine signaling pathway,Adrenergic signaling in cardiomyocytes,African trypanosomiasis,"Alanine, aspartate and glutamate metabolism",Alcoholism,...,Vitamin digestion and absorption,Wnt signaling pathway,alpha-Linolenic acid metabolism,beta-Alanine metabolism,cAMP signaling pathway,cGMP-PKG signaling pathway,mRNA surveillance pathway,mTOR signaling pathway,p53 signaling pathway,uniprot_ids
A2M,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,P01023
A3GALT2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,U3KPV4
A4GALT,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Q9NPC4
AAAS,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Q9NRG9
AACS,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Q86V21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNF98,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,A6NK75
ZNF99,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,A8MXY4
ZNRF3,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,Q9ULT6
ZSCAN32,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Q9NX65


In [31]:
ppi.shape, uniport_kegg_df.shape

((17222, 129), (7377, 309))

In [33]:
len(set(set(ppi['string_id'].tolist())&set(uniport_kegg_df['uniprot_ids'].tolist())))

6728

In [34]:
set(set(uniport_kegg_df['uniprot_ids'].tolist())-set(ppi['string_id'].tolist()))

{'A0A087WWS6',
 'A0A087X1C5',
 'A0A096LP55',
 'A4GXA9',
 'A6NDH6',
 'A6NFI3',
 'A6NGB9',
 'A6NGY5',
 'A6NH00',
 'A6NHX0',
 'A6NI73',
 'A6NIX2',
 'A6NIY4',
 'A6NKK0',
 'A6NL08',
 'A6NM03',
 'A6NMZ5',
 'A6NNF4',
 'A6NNV3',
 'A8K0Z3',
 'A8MPY1',
 'A8MT65',
 'A8MUV8',
 'A8MW95',
 'A8MXY4',
 'A8TX70',
 'O00255',
 'O00303',
 'O00462',
 'O14512',
 'O14514',
 'O14581',
 'O14639',
 'O14788',
 'O14841',
 'O15041',
 'O15042',
 'O15143',
 'O15160',
 'O15197',
 'O15439',
 'O15467',
 'O15554',
 'O43323',
 'O43432',
 'O43508',
 'O43511',
 'O43677',
 'O43826',
 'O43869',
 'O60237',
 'O60404',
 'O60602',
 'O60610',
 'O60907',
 'O75015',
 'O75044',
 'O75123',
 'O75342',
 'O75396',
 'O75582',
 'O75747',
 'O75911',
 'O75970',
 'O76094',
 'O76100',
 'O95045',
 'O95049',
 'O95072',
 'O95136',
 'O95222',
 'O95342',
 'O95544',
 'O95999',
 'P00326',
 'P00338',
 'P00736',
 'P00915',
 'P01042',
 'P01229',
 'P01569',
 'P02549',
 'P02654',
 'P02671',
 'P02746',
 'P04075',
 'P05062',
 'P05106',
 'P05161',
 'P05771'

In [35]:
ppi[ppi['string_id']=='A0A087WWS6']

,string_id,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_119,feature_120,feature_121,feature_122,feature_123,feature_124,feature_125,feature_126,feature_127,feature_128


In [36]:
with open('/itf-fi-ml/shared/users/ziyuzh/svm/data/stringdb/2023/name_convert.pkl', 'rb') as f:
    ppi_name_convert = pickle.load(f)

In [37]:
name2string = ppi_name_convert[1]

In [48]:
# name2string['IFNA13']
name2string['OR5H1']


'9606.ENSP00000492953'

In [40]:
ppi_connection = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/stringdb/2019/9606.protein.links.v10.5.txt', sep=' ', header=0).convert_dtypes().replace(0, float('nan'))

In [41]:
ppi_connection[['protein1', 'protein2']] = np.sort(ppi_connection[['protein1', 'protein2']], axis=1)
ppi_connection = ppi_connection.drop_duplicates()

In [42]:
import networkx as nx

# Convert to NetworkX graph
G = nx.from_pandas_edgelist(ppi_connection, 'protein1', 'protein2')

In [46]:
ppi_2019_nodes = list(G.nodes)

In [49]:
# if '9606.ENSP00000480467' in ppi_2019_nodes:
if '9606.ENSP00000492953' in ppi_2019_nodes:
    print(1)

In [ ]:
# Query mygene for UniProt and Entrez gene ID mappings
temp = mg.querymany(
    [''],
    scopes='symbol',
    fields='uniprot,entrezgene',
    species='human'
)

### get ready

In [ ]:
uniport_kegg_df = uniport_kegg_df.rename(columns={'uniprot_ids': 'string_id'})
ppi

## disease labels

In [ ]:
dga = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/disgent_2020/timecut/dga_time_uniport.csv')
time = 2019

In [ ]:
merge_dga = dga[dga['string_id'].isin(ppi['string_id'])]

selected_diseases = []
dfs = []

for disease_id in merge_dga['disease_id'].unique():
    sub_df = merge_dga[merge_dga['disease_id'] == disease_id]
    
    if len(sub_df) < 15:
        continue
    
    if (
        sub_df['first_pub_year'].max() > time and
        sub_df['first_pub_year'].min() <= time and
        len(sub_df[sub_df['first_pub_year'] < time]) >= 5
    ):
        selected_diseases.append(disease_id)
        
        # label: test = 1 if first_pub_year > time, else 0
        sub_df = sub_df.copy()
        sub_df['label'] = (sub_df['first_pub_year'] > time).astype(int)
        
        dfs.append(sub_df)

# final dataset
final_df = pd.concat(dfs, ignore_index=True)

rows = []

for d in final_df['disease_id'].unique():
    sub_df = final_df[final_df['disease_id'] == d]
    
    all_labels = len(sub_df)
    test_num = (sub_df['label'] == 1).sum()
    train_num = all_labels - test_num
    
    rows.append({
        'disease_id': d,
        'all_labels': all_labels,
        'test_num': test_num,
        'train_num': train_num
    })

summary_df = pd.DataFrame(rows)

In [64]:
merge_dga = dga[dga['string_id'].isin(uniport_kegg_df['string_id'])]

selected_diseases = []
dfs = []

for disease_id in merge_dga['disease_id'].unique():
    sub_df = merge_dga[merge_dga['disease_id'] == disease_id]
    
    if len(sub_df) < 15:
        continue
    
    if (
        sub_df['first_pub_year'].max() > time and
        sub_df['first_pub_year'].min() <= time and
        len(sub_df[sub_df['first_pub_year'] < time]) >= 5
    ):
        selected_diseases.append(disease_id)
        
        # label: test = 1 if first_pub_year > time, else 0
        sub_df = sub_df.copy()
        sub_df['label'] = (sub_df['first_pub_year'] > time).astype(int)
        
        dfs.append(sub_df)

# final dataset
final_df = pd.concat(dfs, ignore_index=True)

rows = []

for d in final_df['disease_id'].unique():
    sub_df = final_df[final_df['disease_id'] == d]
    
    all_labels = len(sub_df)
    test_num = (sub_df['label'] == 1).sum()
    train_num = all_labels - test_num
    
    rows.append({
        'disease_id': d,
        'all_labels': all_labels,
        'test_num': test_num,
        'train_num': train_num
    })

summary_df2 = pd.DataFrame(rows)

In [65]:
len(summary_df),len(summary_df2)

(48, 39)

In [66]:
summary_df['all_labels'].sum(),summary_df2['all_labels'].sum()

(5295, 3529)